# 5주차. Sub-Agent, Skills/MD, 하네스 엔지니어링

## 0. 목표

실제 OpenAI 모델로 supervisor가 요청을 라우팅하고, 나나 또는 카나 agent가 도구를 호출하게 한다.


## 1. 준비


In [ ]:
import json
import os
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".env").exists() or (candidate / ".env.example").exists():
            return candidate
    raise RuntimeError("repo root를 찾지 못했습니다. 노트북을 repo 안에서 실행하세요.")

REPO_ROOT = find_repo_root(Path.cwd())
load_dotenv(REPO_ROOT / ".env", override=False)

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")


def make_model(max_tokens: int = 500) -> ChatOpenAI:
    return ChatOpenAI(model=OPENAI_MODEL, temperature=0, max_completion_tokens=max_tokens)


def show_json(value: Any) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


def final_text(agent_result: dict[str, Any]) -> str:
    return agent_result["messages"][-1].content


def extract_tool_trace(agent_result: dict[str, Any]) -> list[dict[str, Any]]:
    trace = []
    for message in agent_result.get("messages", []):
        for call in getattr(message, "tool_calls", []) or []:
            trace.append({"event": "tool_call", "tool_name": call.get("name"), "arguments": call.get("args", {})})
        if getattr(message, "type", None) == "tool":
            trace.append({"event": "tool_result", "tool_name": getattr(message, "name", None), "content": message.content})
    return trace



## 2. 개념

Sub-agent는 역할과 사용할 도구를 나눈 agent다. Supervisor는 직접 업무 도구를 들고 있지 않고, `nana_agent`, `kana_agent`처럼 sub-agent를 감싼 tool 중 하나를 호출한다.


## 3. 기본 개념 실습

가장 작은 흐름은 supervisor가 개인 요청은 나나에게, 그룹 조율 요청은 카나에게 위임하는 것이다.


In [ ]:
@tool("personal_create_schedule", description="나나가 개인 일정 초안을 만든다.")
def personal_create_schedule(title: str, date: str, start_time: str) -> str:
    """Create a personal schedule."""
    return json.dumps({"ok": True, "schedule": {"title": title, "date": date, "start_time": start_time}}, ensure_ascii=False)

@tool("group_confirm_slot", description="카나가 멤버 응답을 근거로 그룹 시간을 확정한다.")
def group_confirm_slot(topic: str, selected_slot: str, members: list[str], reason: str) -> str:
    """Confirm a group slot from member replies."""
    return json.dumps({"ok": True, "topic": topic, "selected_slot": selected_slot, "members": members, "reason": reason}, ensure_ascii=False)
nana_agent = create_agent(
    model=make_model(600),
    tools=[personal_create_schedule],
    system_prompt="너는 개인 메이트 나나다. 오늘은 2026-04-23이다. 상대 날짜는 이 날짜 기준으로 YYYY-MM-DD로 바꾼다. 개인 일정 요청은 personal_create_schedule 도구를 호출한다.",
)

kana_agent = create_agent(
    model=make_model(700),
    tools=[group_confirm_slot],
    system_prompt="너는 그룹 메이트 카나다. 멤버 응답에서 모두 가능한 시간을 찾으면 group_confirm_slot 도구를 호출한다.",
)

@tool("nana_agent", description="개인 일정 요청을 나나 sub-agent에게 위임한다.")
def delegate_to_nana(request: str) -> str:
    """Delegate a personal request to Nana sub-agent."""
    agent_result = nana_agent.invoke({"messages": [{"role": "user", "content": request}]})
    return json.dumps({"agent": "nana", "answer": final_text(agent_result), "trace": extract_tool_trace(agent_result)}, ensure_ascii=False)

@tool("kana_agent", description="그룹 일정 조율 요청과 멤버 응답을 카나 sub-agent에게 위임한다.")
def delegate_to_kana(request: str, member_replies: str) -> str:
    """Delegate a group coordination request to Kana sub-agent."""
    message = f"요청: {request}\n멤버 응답:\n{member_replies}"
    agent_result = kana_agent.invoke({"messages": [{"role": "user", "content": message}]})
    return json.dumps({"agent": "kana", "answer": final_text(agent_result), "trace": extract_tool_trace(agent_result)}, ensure_ascii=False)

supervisor = create_agent(
    model=make_model(900),
    tools=[delegate_to_nana, delegate_to_kana],
    system_prompt="너는 카나메이트 supervisor다. 개인 일정 요청은 nana_agent tool을 호출하고, 그룹 일정 조율 요청은 kana_agent tool을 호출한다. 직접 처리하지 말고 반드시 적절한 sub-agent tool을 호출한 뒤 그 결과를 학생에게 요약한다.",
)

def delegated_agent_from_trace(agent_result: dict[str, Any]) -> str:
    for event in extract_tool_trace(agent_result):
        if event.get("event") == "tool_call" and event.get("tool_name") in {"nana_agent", "kana_agent"}:
            return event["tool_name"].replace("_agent", "")
    return "unknown"


def delegated_payload_from_trace(agent_result: dict[str, Any]) -> dict[str, Any]:
    for event in extract_tool_trace(agent_result):
        if event.get("event") == "tool_result" and event.get("tool_name") in {"nana_agent", "kana_agent"}:
            return json.loads(event["content"])
    return {}


## 4. 카나메이트 확장 예제

라우팅을 눈으로 한 번만 확인하지 않고 `run_supervisor_case` 하네스로 감싼다. 이 하네스는 선택된 sub-agent와 내부 tool trace를 한 report로 만든다.


In [ ]:
def run_supervisor_case(case: dict[str, Any]) -> dict[str, Any]:
    """Run one routing case and return a compact report."""
    content = case["request"]
    if case.get("member_replies"):
        content = f"요청: {case['request']}\n멤버 응답:\n{case['member_replies']}"

    result = supervisor.invoke({"messages": [{"role": "user", "content": content}]})
    selected_agent = delegated_agent_from_trace(result)
    delegate_payload = delegated_payload_from_trace(result)
    inner_tool_names = [
        event["tool_name"]
        for event in delegate_payload.get("trace", [])
        if event.get("event") == "tool_call"
    ]
    return {
        "name": case["name"],
        "expected_agent": case.get("expected_agent"),
        "selected_agent": selected_agent,
        "expected_inner_tool": case.get("expected_inner_tool"),
        "inner_tool_names": inner_tool_names,
        "answer": final_text(result),
        "trace": extract_tool_trace(result),
        "delegate_payload": delegate_payload,
    }

extension_case = {
    "name": "group_slot",
    "request": "팀 회의 시간을 조율해줘",
    "member_replies": "민수: 2026-04-24 10:00 가능\n지아: 2026-04-24 10:00 가능\n준호: 2026-04-24 10:00 가능",
    "expected_agent": "kana",
    "expected_inner_tool": "group_confirm_slot",
}
extension_result = run_supervisor_case(extension_case)

show_json({"selected_agent": extension_result["selected_agent"], "inner_tool_names": extension_result["inner_tool_names"]})
print(extension_result["answer"])
show_json(extension_result["trace"])


## 5. 확장 예제 실행

같은 하네스로 개인 일정 케이스와 그룹 조율 케이스를 연속 실행한다. 학생은 케이스 입력을 바꿔 라우팅 결과와 내부 tool 이름을 비교한다.


In [ ]:
extension_cases = [
    {
        "name": "personal_schedule",
        "request": "내일 9시에 민수와 1:1 일정 잡아줘",
        "member_replies": "",
        "expected_agent": "nana",
        "expected_inner_tool": "personal_create_schedule",
    },
    {
        "name": "group_slot",
        "request": "팀 회의 시간을 조율해줘",
        "member_replies": "민수: 2026-04-25 15:00 가능\n지아: 2026-04-25 15:00 가능",
        "expected_agent": "kana",
        "expected_inner_tool": "group_confirm_slot",
    },
]
extension_results = [run_supervisor_case(case) for case in extension_cases]

show_json([
    {
        "name": result["name"],
        "expected_agent": result["expected_agent"],
        "selected_agent": result["selected_agent"],
        "expected_inner_tool": result["expected_inner_tool"],
        "inner_tool_names": result["inner_tool_names"],
    }
    for result in extension_results
])

for result in extension_results:
    assert result["selected_agent"] == result["expected_agent"]
    assert result["expected_inner_tool"] in result["inner_tool_names"]
print("5주차 확장 예제 실행 통과")


## 6. 코드 작성형 실습(`week05.py`)

이번 실습은 supervisor 하네스와 Golden Scenario를 같은 주차 과제 파일 `week05.py`의 `run_supervisor_case`에서 구현하고 실행한다.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "week05.py").exists():
            return candidate
    raise RuntimeError("repo root를 찾지 못했습니다. 노트북을 repo 안에서 실행하세요.")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from week05 import extract_tool_trace, final_text, show_json, golden_cases, run_supervisor_case


### 실행 예시

In [ ]:
golden_results = [run_supervisor_case(case) for case in golden_cases]
show_json([
    {
        "name": result["name"],
        "expected_agent": result["expected_agent"],
        "selected_agent": result["selected_agent"],
        "expected_inner_tool": result["expected_inner_tool"],
        "inner_tool_names": result["inner_tool_names"],
    }
    for result in golden_results
])

## 6-0. 실습 자동 점검

아래 셀은 모델 답변 문구가 아니라 trace, structured response, payload처럼 구조화된 값을 확인한다.

In [ ]:
assert len(golden_results) == len(golden_cases)
for result in golden_results:
    assert result["selected_agent"] == result["expected_agent"]
    assert result["expected_inner_tool"] in result["inner_tool_names"]
print("5주차 코드 작성형 실습 통과")

## 6-1. 로컬 Gradio UI 실습

이번 주 Gradio UI도 `week05.py` 안에 들어 있다. 터미널에서 아래 명령을 실행하면 해당 주차 UI가 바로 열린다.

```bash
python week05.py
```

노트북 안에서는 다음 셀에서 `create_demo()`로 Gradio Blocks 객체를 확인할 수 있다.

In [ ]:
from week05 import create_demo

demo = create_demo()
demo

## 7. 회고

이번 주에는 역할이 다른 agent를 나누고, 실제 모델 라우팅과 tool call trace로 동작을 확인했다.
